# Milestone 4 — Team assignment: clustering shirt colours

**Problem after Milestone 3:** every player has a stable ID, but the detector only says `player`. We do not know which team anyone plays for, so possession, team stats and the 2D map are impossible.

**Fix:** players of one team wear the same shirt. Take the shirt colour of every player, let **KMeans** split the colours into 2 groups, then decide each **track's** team by a vote over its frames.

**Input:** `outputs/m3_tracks.csv` (from Milestone 3) and the video. No detector, no GPU needed in this notebook.

Plan:
1. Concepts
2. Setup and load the track table
3. From a player box to one colour (step by step, with pictures)
4. Collect colours for every track
5. KMeans with k = 2, and how good the split is
6. Look at the teams (crop galleries)
7. One team per track: the vote
8. Goalkeepers and referees
9. Choose which team is which (name them)
10. Experiments: colour spaces and shirt region
11. Render the video and save the table
12. Check yourself

## 1. Concepts

**Unsupervised learning.** Until now every model had labels (boxes drawn by humans). Here we have **no labels**: nobody told us which shirt is team A. **KMeans** finds groups by itself: it places k centre points and moves them until each point is close to its own centre. You already know it from scikit-learn; here each "row" is one player crop and the "features" are 3 colour numbers.

**Why only the shirt.** A player box contains head, shirt, shorts, socks, legs and grass. The shirt is the part that differs between teams, so we cut out the **upper middle** of the box: below the head, above the shorts, away from the arms.

**Why remove grass, and how.** Even the shirt crop has grass pixels between the arms or around a thin player. Grass is the same for both teams, so it only blurs the difference.
- *First idea (fixed range):* drop every pixel in the green HSV range of the pitch mask. **This breaks when a team wears green**: their shirts are deleted as "grass", their colour comes from skin and shorts, and some of them land in the other team.
- *What we use (corners):* the four corners of a player box are almost always pitch. We measure **this player's own grass colour** at the corners and remove only pixels close to it. A green shirt survives as long as it differs from the grass right next to it.

**One colour per crop: the median.** The median of the remaining pixels is robust: a few skin or number pixels do not move it, unlike the mean.

**Colour spaces** (three ways to write the same colour as 3 numbers):
| Space | Numbers | Good for | Problem |
|---|---|---|---|
| BGR | blue, green, red amounts | what OpenCV reads | brightness and colour are mixed: a shirt in shadow looks like a different colour |
| HSV | hue, saturation, value | picking a colour range (grass) | **hue is a circle**: red is both 0 and 180, so two red shirts can look maximally different |
| **Lab** | lightness L, green–red a, blue–yellow b | **comparing colours**: distances roughly match what the eye sees | none for us; this is our default |

**Per-track vote.** A single crop can be wrong (shadow, an overlapping opponent, motion blur). Each track has many crops, so the team of a track = the team most of its crops got. Same idea as the class vote in Milestone 3.

**Goalkeepers and referees.** Goalkeepers wear their own colour, so colour would make them a third team. We assign them by **position** instead (a goalkeeper stands near his own team's half). Referees get **no team**.

**Which team is which?** KMeans numbers clusters 0 and 1 at random. We re-order them so that **team 0 is always the lighter shirt**, then you look at the colours and give them names.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from football_tracking.teams import (ShirtRegion, shirt_crop, shirt_pixels, non_grass_pixels, colour_features, sample_rows,
    collect_features, TeamClassifier, vote_tracks, assign_goalkeepers, add_team_column, team_balance,
    TeamAnnotator, swatch, SPACES, GRASS_LOWER, GRASS_UPPER)
from football_tracking.video_io import video_info, iter_frames, VideoWriter

CLIP_NAME = "08fd33_0.mp4"
VIDEO = ROOT / "data" / "raw" / CLIP_NAME
TRACKS = ROOT / "outputs" / "m3_tracks.csv"
OUT = ROOT / "outputs"
assert VIDEO.exists(), VIDEO
assert TRACKS.exists(), f"{TRACKS} missing: run section 9 of notebook 03 first"

df = pd.read_csv(TRACKS)
info = video_info(VIDEO)
CLS = "voted_class_name"                     # the class decided by the vote in Milestone 3
print(info)
print(df[df.tracker_id >= 0].groupby(CLS).tracker_id.nunique().rename("tracks"))
df.head()

## 2. From a player box to one colour, step by step

Pick one player and watch each step. Change `ROW` to try others (try one in shadow and one next to an opponent).

In [ ]:
def read_frame(i):
    cap = cv2.VideoCapture(str(VIDEO)); cap.set(cv2.CAP_PROP_POS_FRAMES, i)
    ok, fr = cap.read(); cap.release(); assert ok; return fr

players = df[(df.tracker_id >= 0) & (df[CLS] == "player")].copy()
players["h"] = players.y2 - players.y1
players = players[players.h.between(players.h.quantile(.4), players.h.quantile(.7))]   # typical height, not the extremes
SEED = 0                                                         # change to see other players

fe = None
for ROW in players.sample(frac=1, random_state=SEED).index:      # try players until one has enough shirt pixels
    r = df.loc[ROW]
    frame = read_frame(int(r.frame))
    box_xyxy = (r.x1, r.y1, r.x2, r.y2)
    px, shown_corners = shirt_pixels(frame, box_xyxy, method="corners")
    fe = colour_features(px)
    if fe is not None:
        break
assert fe is not None, "no player with enough shirt pixels"
x1, y1, x2, y2 = map(int, box_xyxy)
box = frame[y1:y2, x1:x2]
crop = shirt_crop(frame, box_xyxy)
_, shown_hsv = shirt_pixels(frame, box_xyxy, method="hsv")
median_patch = np.full((60, 60, 3), [fe["B"], fe["G"], fe["R"]], np.uint8)

fig, axs = plt.subplots(1, 5, figsize=(14, 4))
for ax, img, t in zip(axs, [box, crop, shown_hsv, shown_corners, median_patch],
                      ["1. player box", "2. shirt region", "3a. fixed green range\n(removed = white)",
                       "3b. corners method\n(removed = white)", "4. median colour (3b)"]):
    if img.size: ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(t, fontsize=9); ax.axis("off")
plt.suptitle(f"track #{int(r.tracker_id)}, frame {int(r.frame)}  |  Lab = ({fe['L']:.0f}, {fe['A']:.0f}, {fe['Bb']:.0f}), "
             f"{fe['n_pixels']} pixels kept"); plt.tight_layout(); plt.show()

**What to check:** the shirt region (2) should be mostly shirt. If you see a lot of shorts, lower `bottom`; if you see the head, raise `top` (see `ShirtRegion` in `teams.py`). Compare 3a and 3b: for a **green-shirted** player, 3a turns most of the shirt white (wrong), 3b keeps it. Change `SEED` until you see a player from each team.

## 3. Collect one colour per crop, for every track

We do not need every frame of every player (that would be ~15,000 crops). For each track we take **up to 20 crops**, chosen at random over its life, but only when:
- the box is at least 40 px tall (tiny far-away players have only a few shirt pixels),
- it does not overlap another person (IoU ≤ 0.1), so we do not mix two shirts.

Players who are **never** alone get a few of their least-overlapped crops, marked `fallback`. Those are predicted with the model but not used to train it.

In [ ]:
rows = sample_rows(df, class_col=CLS, classes=("player",), per_track=20)
print(len(rows), "crops from", rows.tracker_id.nunique(), "player tracks |",
      int(rows.fallback.sum()), "fallback crops")
feats, sample_crops = collect_features(VIDEO, rows, keep_crops=400)   # reads the video once
print(feats.shape)
feats.head()

**Health check:** `kept_frac` is the share of the shirt region that survived background removal. If one team keeps far less than the other, background removal is eating their shirts (that was the green-team bug).

In [ ]:
quick = TeamClassifier().fit(feats)
print(feats.assign(team=quick.predict(feats)).groupby("team").kept_frac.describe()[["count", "25%", "50%", "75%"]].round(2))

## 4. KMeans with k = 2

`TeamClassifier` fits KMeans on the Lab colours of the clean crops, re-orders the clusters (team 0 = lighter shirt) and stores each team's typical shirt colour.

**Silhouette score** (from scikit-learn) measures how well separated the groups are: near **1** = two clear groups, near **0** = groups overlap, negative = many points in the wrong group. Two clearly different kits usually give 0.5 or more.

In [ ]:
tc = TeamClassifier(space="lab").fit(feats)
labels = tc.predict(feats)
print("silhouette:", round(tc.silhouette, 3))
print("crops per team:", pd.Series(labels).value_counts().sort_index().to_dict())
print("team colours (BGR):", tc.colours_bgr)
plt.figure(figsize=(4, 2.4)); plt.imshow(cv2.cvtColor(swatch(tc.colours_bgr, {0: "team 0", 1: "team 1"}), cv2.COLOR_BGR2RGB))
plt.axis("off"); plt.show()

### See the clusters

Each dot is one crop, drawn **in its own median colour**, placed by its Lab numbers. The black crosses are the KMeans centres. Two clouds = an easy split. Dots between the clouds are the crops to worry about.

Note: OpenCV stores Lab in 0 to 255 for 8-bit images, so **a = 128 and b = 128 mean neutral** (grey, white, black). White and black kits sit near (128, 128) and differ only in L; coloured kits sit far from 128.

In [ ]:
X = feats[["L", "A", "Bb"]].to_numpy(float)
dot_rgb = feats[["R", "G", "B"]].to_numpy() / 255
C = tc.km.cluster_centers_
fig, axs = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, (i, j, xl, yl) in zip(axs, [(1, 2, "a  (green  ←→  red)", "b  (blue  ←→  yellow)"), (1, 0, "a  (green  ←→  red)", "L  (dark  ←→  light)")]):
    ax.scatter(X[:, i], X[:, j], c=dot_rgb, s=28, edgecolor=np.where(labels == 0, "#444", "#999"), linewidth=.6)
    ax.scatter(C[:, i], C[:, j], marker="x", s=160, c="black", linewidth=3)
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.grid(alpha=.25)
plt.suptitle("Every crop in Lab colour space (dot colour = the crop's real median colour)"); plt.tight_layout(); plt.show()

## 5. Look at the teams

Numbers are not enough: look at the crops. Each row should show one kit. A crop in the wrong row tells you what fooled the model (shadow, an opponent in the crop, a goalkeeper or referee wrongly voted as `player`).

In [ ]:
def gallery(crops, crop_labels, team, n=24, size=(48, 64)):
    pick = [c for (tid, c), l in zip(crops, crop_labels) if l == team][:n]
    if not pick: return np.zeros((size[1], size[0], 3), np.uint8)
    tiles = [cv2.resize(c, size) for c in pick]
    while len(tiles) % 12: tiles.append(np.full_like(tiles[0], 255))
    rows_ = [np.hstack(tiles[k:k + 12]) for k in range(0, len(tiles), 12)]
    return np.vstack(rows_)

# team label of every stored crop: feats.crop_idx tells which stored crop each feature row belongs to
stored = feats.crop_idx.to_numpy() >= 0
crop_labels = np.empty(len(sample_crops), int)
crop_labels[feats.crop_idx.to_numpy()[stored]] = labels[stored]
fig, axs = plt.subplots(2, 1, figsize=(12, 5))
for ax, t in zip(axs, [0, 1]):
    ax.imshow(cv2.cvtColor(gallery(sample_crops, crop_labels, t), cv2.COLOR_BGR2RGB)); ax.set_title(f"team {t}"); ax.axis("off")
plt.tight_layout(); plt.show()

## 6. One team per track: the vote

`agreement` = share of a track's crops that voted for the winning team. **1.0** = every crop agrees. Low agreement (< 0.7) means something is off for that track; the most common reasons:
- an **ID switch** between two players of different teams (Milestone 3 issue),
- the track is really a **referee or goalkeeper** labelled `player`,
- the player was always in a **crowd** (fallback crops).

In [ ]:
votes = vote_tracks(feats, labels)
print(votes.team.value_counts().sort_index().rename("tracks per team").to_dict())
print("median agreement:", votes.agreement.median().round(2))
votes.sort_values("agreement").head(10)

In [ ]:
# look at the least certain track: its crops, and which team each crop voted for
tid = int(votes.sort_values("agreement").iloc[0].tracker_id)
own = [(c, l) for (t, c), l in zip(sample_crops, crop_labels) if t == tid][:12]
if own:
    fig, axs = plt.subplots(1, len(own), figsize=(1.3 * len(own), 2.4))
    for ax, (c, l) in zip(np.atleast_1d(axs), own):
        ax.imshow(cv2.cvtColor(c, cv2.COLOR_BGR2RGB)); ax.set_title(f"team {l}", fontsize=8); ax.axis("off")
    plt.suptitle(f"track #{tid}: crops and their votes"); plt.tight_layout(); plt.show()
else:
    print("no stored crops for this track (keep_crops limit); raise keep_crops in section 3 to see it")

## 7. Goalkeepers and referees

**Goalkeepers:** in every frame where a goalkeeper is visible, compare his x position with the average x of each team's outfield players; the closer team gets a vote. This is a **pixel** approximation (the camera moves), so check the result by eye. Milestone 6 (pitch coordinates) will make it exact.

**Referees:** no team (`-1`).

In [ ]:
team_of = dict(zip(votes.tracker_id, votes.team))
gk_ids = df[(df.tracker_id >= 0) & (df[CLS] == "goalkeeper")].tracker_id.unique()
gk_team = assign_goalkeepers(df, team_of, gk_ids, class_col=CLS)
print("goalkeepers:", gk_team)
team_of.update(gk_team)
dft = add_team_column(df, team_of)
bal = team_balance(dft, class_col=CLS)
print("players per team per frame (mean):", bal.mean().round(1).to_dict())
bal.plot(figsize=(11, 2.8), title="players per team in each frame"); plt.xlabel("frame"); plt.show()

**Reading the balance plot.** When the camera shows the whole pitch both lines should be near 10. When it zooms on one area they can differ a lot, which is normal. A team that is **always** far bigger suggests referees or the other team's players were put into it.

## 8. Choose which team is which

Team 0 is always the lighter shirt (so the numbers stay the same when you re-run). Look at the swatch and the galleries above, then write the names. For a match with known kits you would write the club names here.

In [ ]:
TEAM_NAMES = {0: "Green", 1: "White"}        # sample clip: team 0 (lighter shirt) is the lime-green team; edit for other clips
plt.figure(figsize=(4, 2.4)); plt.imshow(cv2.cvtColor(swatch(tc.colours_bgr, TEAM_NAMES), cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()

## 9. Experiments

Change **one** thing at a time and compare. We judge with three numbers: silhouette (separation), median track agreement, and how many tracks have agreement under 0.7.

In [ ]:
def evaluate(feats_, space):
    t = TeamClassifier(space=space).fit(feats_)
    v = vote_tracks(feats_, t.predict(feats_))
    return {"silhouette": round(t.silhouette, 3), "median_agreement": round(v.agreement.median(), 3),
            "tracks_below_0.7": int((v.agreement < 0.7).sum()), "tracks": len(v)}

print("A. colour space (same crops):")
pd.DataFrame({sp: evaluate(feats, sp) for sp in ["lab", "bgr", "hsv"]}).T

In [ ]:
print("B. background removal method (re-reads the video for each, ~1 min each):")
res = {}
for m in ["corners", "hsv", "none"]:
    f_, _ = collect_features(VIDEO, rows, method=m)
    res[m] = evaluate(f_, "lab"); res[m]["crops_kept"] = len(f_)
pd.DataFrame(res).T

In [ ]:
print("C. shirt region (re-reads the video for each setting, ~1 min each):")
regions = {
    "default 15-50% tall, 25-75% wide": ShirtRegion(),
    "whole box":                       ShirtRegion(0.0, 1.0, 0.0, 1.0),
    "upper half, full width":          ShirtRegion(0.0, 0.5, 0.0, 1.0),
    "narrow chest 20-45%, 35-65%":     ShirtRegion(0.20, 0.45, 0.35, 0.65),
}
res = {}
for name, reg in regions.items():
    f_, _ = collect_features(VIDEO, rows, region=reg)
    res[name] = evaluate(f_, "lab")
pd.DataFrame(res).T

**What to expect**
- **hsv (fixed green range)** fails when a team wears green: fewer crops kept, lower agreement, a lower silhouette.
- **none** keeps grass, so the two teams' colours are pulled towards green and towards each other.
- **Whole box** is usually worst: shorts, socks and grass are shared between teams and blur the difference.
- **HSV** can be poor when a team wears red (hue wraps around) or white/black (hue is meaningless for grey colours).
- Lab and BGR are often close on clear kits; Lab is more stable when shadows fall across part of the pitch.

## 9b. Repair ID switches, then check the referees

Two problems found on the real clip:
1. **ID switch between roles.** Track #15 was the referee for 680 frames, then its ID jumped onto a white player. The whole-track vote called that player `REF`. Fix: split a track where the detector's class changes for at least 25 frames in a row (short flicker never splits), then re-vote and give new player tracks a team.
2. **A player called `referee` in every frame** cannot be fixed by voting. Colour can: a referee track whose shirt is closer to a team colour than to the main referee is relabelled as a player of that team.

In [ ]:
from football_tracking.tracking import split_on_class_change

dfs, log = split_on_class_change(dft)
print("splits (old id, new id, from frame):", log)

# re-vote the class of every track (confidence-weighted, like Milestone 3)
people = dfs.tracker_id >= 0
best = (dfs[people].groupby(["tracker_id", "class_name"]).conf.sum().reset_index()
        .sort_values("conf", ascending=False).drop_duplicates("tracker_id").set_index("tracker_id").class_name)
dfs.loc[people, CLS] = dfs.loc[people, "tracker_id"].map(best)

# team for the NEW tracks: players by shirt colour, goalkeepers by position
new_ids = [new for _, new, _ in log]
new_players = [t for t in new_ids if best.get(t) == "player"]
if new_players:
    nr = sample_rows(dfs, class_col=CLS, classes=("player",))
    nf, _ = collect_features(VIDEO, nr[nr.tracker_id.isin(new_players)])
    if len(nf):
        nv = vote_tracks(nf, tc.predict(nf))
        team_of.update(dict(zip(nv.tracker_id, nv.team)))
new_gks = [t for t in new_ids if best.get(t) == "goalkeeper"]
if new_gks:
    outfield = {t: v for t, v in team_of.items() if best.get(t) == "player"}
    team_of.update(assign_goalkeepers(dfs, outfield, new_gks, class_col=CLS))
dfs["team"] = dfs.tracker_id.map(team_of).fillna(-1).astype(int)
dfs.loc[dfs[CLS] == "referee", "team"] = -1
dft = dfs

In [ ]:
from football_tracking.teams import check_referee_tracks, apply_referee_check

check = check_referee_tracks(VIDEO, dft, tc, class_col=CLS)
dft = apply_referee_check(dft, check, class_col=CLS)
check            # verdict 'player' = relabelled; on the sample clip every referee track stayed a referee

## 10. Render the video and save the table

In [ ]:
ann = TeamAnnotator(tc.colours_bgr, TEAM_NAMES)
by_frame = {f: g for f, g in dft.groupby("frame")}
out_path = OUT / "m4_teams.mp4"
with VideoWriter(out_path, info["fps"], info["width"], info["height"]) as w:
    for i, frame in iter_frames(VIDEO):
        g = by_frame.get(i)
        w.write(ann(frame, g, class_col=CLS) if g is not None else frame)
dft.to_csv(OUT / "m4_tracks_teams.csv", index=False)
print("saved", out_path, "and", OUT / "m4_tracks_teams.csv")

In [ ]:
# a quick look at one frame of the result
i = int(dft.frame.median())
plt.figure(figsize=(14, 8)); plt.imshow(cv2.cvtColor(ann(read_frame(i), by_frame[i], class_col=CLS), cv2.COLOR_BGR2RGB))
plt.axis("off"); plt.title(f"frame {i}"); plt.show()

## 11. Check yourself

**Questions** (answer on paper first):
1. Why is this unsupervised learning, and what plays the role of a "row" and of "features"?
2. Why do we crop only the upper middle of the box?
3. Why the median colour, not the mean?
4. Why is HSV a bad space to cluster red shirts in?
5. Why is the team decided per track and not per crop?
6. Why are goalkeepers assigned by position, and why is that only approximate here?
7. KMeans numbers clusters at random. How do we keep "team 0" meaning the same team between runs?

**Do it yourself**
- Watch `m4_teams.mp4`. Write down any player whose ellipse has the wrong colour, with the frame number, and look up his track in the `votes` table.
- Fill in: silhouette ____ | median agreement ____ | tracks below 0.7 ____ | goalkeepers correct? ____

**Pitfalls**
- **BGR vs RGB**: plotting without `cv2.cvtColor(..., COLOR_BGR2RGB)` swaps red and blue, so you would misname the teams.
- **Tracks voted as `player` that are really referees** become a third colour inside one team. Check the galleries; a bad vote is a Milestone 2/3 issue.
- **Similar kits** (e.g. dark blue vs black) give a low silhouette. Then try the narrow chest region, or more crops per track.
- **Green kits:** never remove background with a fixed green range; compare with the player's own surroundings (corners method). If shirts still disappear, lower `threshold` in `shirt_pixels` (default 18).
- **Floodlight shadows** split one kit into light and dark. Lab helps; voting per track helps more.